# 06 — Đánh giá Algo 1: Semantic Retrieval + History-aware Visual Diversity

Đặc tả: `docs/algo_1_semantic_retrieval_history_diversity.md`. Code: `src/topicx/algo1.py`.

Mỗi vòng lấy top-K = 500 trong phần pool còn lại, lọc theo τ, rồi greedy farthest-first chọn B = 16 ảnh, tính khoảng cách tới H ∪ S. Batch được chấp nhận toàn bộ (mô phỏng) rồi đưa vào lịch sử.
Ngân sách tích lũy là bội số của 16. Selection không phụ thuộc kết quả train, nên tập ngân sách 128 chính là 8 vòng đầu của tập ngân sách 256. Mỗi K vì vậy là một điểm "sau N vòng".

| Nhánh | Câu hỏi (§9) |
|---|---|
| RANDOM | Retrieval và selection có tốt hơn lấy ngẫu nhiên không? |
| ALGO1_TOPB | top-B relevance trong tập đủ ngưỡng (đối chứng không diversity) |
| ALGO1_BATCH | greedy diversity chỉ trong batch: diversity nội batch có ích không? |
| ALGO1 | greedy với H ∪ S: xét lịch sử có thêm ích lợi không? |
| REPLAY_ONLY | đối chứng: mở rộng head + train tiếp, không có dữ liệu mới |

Cách chọn khi triển khai (đặc tả để ngỏ):
- τ chọn trên dev bằng luật F1 khai báo trước (`algo1.tau: dev_f1`).
- Relevance là điểm topic CLIP (max theo 3 crop, prompt chọn trên dev).
- v_x là CLIP image embedding (trung bình 3 crop).
- H_0 rỗng.
- Thiếu ứng viên thì nhân đôi K, không hạ τ.

`SCENARIO` ở cell setup: `missing` hoặc `weak`. Gắn output NB00 của cùng kịch bản để dùng lại splits và base model.

**Hết giờ session / chạy tiếp:** Save Version → gắn output version vừa xong làm input của chính notebook này → chạy lại.
Mọi bước đã có kết quả được bỏ qua (manifest, base model, run, eval đều cache).

In [ ]:
# === Setup (giống nhau ở mọi notebook) ===
SMOKE = True   # True: pipeline thật trên tập nhỏ. Run thật: False
SCENARIO = "missing"   # missing: base chưa có novel class | weak: base có ít ảnh novel
REPO_URL = "https://github.com/Le-Anh-Duy/topicx-yolo-expansion.git"
COMMIT = "main"   # run thật: pin commit hash
import os, subprocess, sys
R = "/tmp/repo"
if not os.path.exists(R):
    subprocess.run(["git", "clone", "-q", REPO_URL, R], check=True)
subprocess.run(["git", "-C", R, "fetch", "-q"], check=True)
subprocess.run(["git", "-C", R, "checkout", "-q", COMMIT], check=True)
subprocess.run([sys.executable, "-m", "pip", "install", "-q", "-r", f"{R}/requirements.txt"], check=True)
sys.path.insert(0, f"{R}/src")
import numpy as np, pandas as pd
from topicx import common as C, pipeline as P
cfg = C.load_config(SMOKE, SCENARIO)
print(SCENARIO, cfg["art"], "| copy từ input:", C.sync_inputs(cfg), "file")

In [ ]:
STAGE = "algo1"
# Test protocol (CPU, vài giây). Lỗi thì dừng notebook.
subprocess.run([sys.executable, "-m", "pytest", "-q", f"{R}/tests"], check=True)
C.env_report(cfg, STAGE);

## Health check dataset
Vài giây, không đọc cả file nhãn: đường dẫn, số ảnh, định dạng nhãn, ảnh mẫu có vẽ box. Có mục FAIL thì notebook dừng ở đây.

In [ ]:
hc, sample = P.health_check(cfg)
if sample is not None:
    display(sample.resize((640, 360)))   # box đỏ phải ôm đúng vật thể

In [ ]:
NOVEL, REASON = "bus", "EDA ở NB1"
SEEDS = cfg["seeds"]
B = cfg["algo1"]["B"]
KS = [2 * B] if SMOKE else [8 * B, 16 * B]   # ngân sách tích lũy: 8 và 16 vòng
BUILTIN = ["RANDOM", "ALGO1_TOPB", "ALGO1_BATCH", "ALGO1"]
REFERENCE = ["REPLAY_ONLY"]
PROPOSERS = {b: P.PROPOSERS[b] for b in BUILTIN}
BRANCHES = BUILTIN + REFERENCE
assert all(k % B == 0 for k in KS)
print(BRANCHES, SEEDS, KS, cfg["algo1"])

## 1–2. Splits và base model (dùng lại nếu đã có trong input)

In [ ]:
meta = P.prepare_splits(cfg, NOVEL, REASON)
W, bm = P.train_base(cfg)
display(pd.DataFrame(bm["dev_ap"]).T)
print(f"ước tính train: {P.estimate_hours(cfg, len(BRANCHES), SEEDS, KS):.1f} h GPU")

## 3. Proposal nhiều vòng → chốt manifest → simulated annotation

In [ ]:
pool = P.Pool(cfg)
P.propose(cfg, PROPOSERS, SEEDS, KS, pool)
print("τ:", C.load(C.out(cfg, "retrieval/tau.json")))
ret = P.retrieval_eval(cfg, pool)
display(ret.groupby(["k", "branch"])[["precision_at_k", "recall_at_k", "novel_instances", "base_instances", "nn_cos_mean"]].mean())

In [ ]:
# Theo từng vòng: separation (proxy) và positive yield thực tế (mở nhãn SAU khi manifest đã chốt)
from topicx.oracle import OracleStore
oracle = OracleStore(C.WORK / cfg["art"] / "splits" / "oracle", C.out(cfg, "retrieval/oracle_log.jsonl"))
rows = []
for p in sorted((C.WORK / cfg["art"] / "selections").glob("*.json")):
    m = C.read_frozen(p)
    if "rounds" not in m or m["k"] != max(KS):
        continue
    bx, _ = oracle.reveal(p)
    pos = set(bx.image[bx.cls == NOVEL])
    for r in m["rounds"]:
        batch = m["ids"][r["round"] * B:(r["round"] + 1) * B]
        rows.append({"branch": m["branch"], "seed": m["seed"], **r, "pos_images": sum(i in pos for i in batch)})
rt = pd.DataFrame(rows)
display(rt.groupby(["round", "branch"])[["min_pair_d", "min_hist_d", "rel_mean", "pos_images"]].mean().unstack("branch").round(3))

## 4. Finetune từng nhánh

In [ ]:
todo = P.train_branches(cfg, BRANCHES, SEEDS, KS)
print("còn thiếu (chạy tiếp ở version sau):", todo or "không")

## 5. Đánh giá trên final test

In [ ]:
res = P.evaluate(cfg, BRANCHES, SEEDS, KS)
print(f"base model: base mAP50-95 {res['base_map']:.4f}, AP novel {res['base_novel_ap']:.4f} | thiếu run: {res['missing'] or 'không'}")
display(res["summary"])
# So sánh theo cặp của §9, cùng seed và cùng ngân sách tích lũy
df = res["runs"]
pairs = [("ALGO1", "ALGO1_BATCH", "giá trị của lịch sử"), ("ALGO1_BATCH", "ALGO1_TOPB", "giá trị của diversity"),
         ("ALGO1_TOPB", "RANDOM", "giá trị của retrieval + τ"), ("ALGO1", "RANDOM", "Algo 1 vs random")]
for col in ("novel_ap50_95", "base_map_delta"):
    piv = df.pivot_table(index=["k", "seed"], columns="branch", values=col)
    d = pd.DataFrame({f"{a} − {b} ({why})": piv[a] - piv[b] for a, b, why in pairs if a in piv and b in piv})
    print(col); display(d.round(4)); display(d.groupby("k").agg(["mean", "std"]).round(4))

## Đọc kết quả (theo §9 của đặc tả)

- **Lịch sử có ích không:** `ALGO1 − ALGO1_BATCH` về AP novel, ở cùng ngân sách.
- **Diversity có ích không:** `ALGO1_BATCH − ALGO1_TOPB`.
- **Retrieval + τ có ích không:** `ALGO1_TOPB − RANDOM`.
- **Bác bỏ giả thuyết** nếu redundancy giảm (`min_pair_d` / `min_hist_d` tăng, `nn_cos_mean` giảm) mà AP không tăng, nếu positive yield giảm quá nhiều, hoặc nếu `base_map_delta` xấu hơn REPLAY_ONLY quá giới hạn.

Distance tăng chỉ cho thấy proxy được tối ưu tốt hơn. Kết luận về chất lượng dữ liệu phải dựa trên AP downstream. ALGO1, ALGO1_BATCH và ALGO1_TOPB tất định theo seed, nên phương sai của chúng chỉ đến từ seed train.